# From Poison to Antidote: Advancing Cybersecurity Education with AI Attack and Defense Training

⚠️Please run the first cell to load the dataset🗂️

In [ ]:
!git clone https://github.com/Exercise-Dataset/Exercise_Dataset.git


Cloning into 'Exercise_Dataset'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 14.77 MiB | 8.31 MiB/s, done.


***OFFENSIVE EXERCISES***

🔎Q1: Run the poisoning attack for different poisoning rates ranging from 10% to 50% with step 10 retraining the model to calculate the performance metrics.🧪📊

*Listing 1*

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


input_file_path="/content/Exercise_Dataset/exercise.csv"
dataset = pd.read_csv(input_file_path, sep=",", decimal=".")
# Prepare data for training and testing
X = dataset[['Tot size', 'syn_flag_number','Variance', 'TCP', 'ICMP', 'ack_flag_number','Header_Length', 'psh_flag_number', 'IAT', 'UDP','fin_flag_number', 'rst_flag_number', 'Std']]
y = dataset['Mapped_Label'].astype('category')
class_names = y.cat.categories.tolist()

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_indices = X_train.index
test_indices = X_test.index

# Initialize list for results

predictions = []
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []


poisoning_rate = np.arange(0, 0.6, 0.1)
# Run through different poisoning rate percentages
for n in poisoning_rate:
    poisoned_dataset = dataset.copy()
    # Select all the ICMP_FLOOD instances in training set
    ICMP_FLOOD_train_instances = poisoned_dataset[(poisoned_dataset["Mapped_Label"] == "ICMP_FLOOD") &
    (poisoned_dataset.index.isin(train_indices))].index.tolist()

    # Find the proportion of the IMCP_FLOOD class to be flipped
    ICMP_FLOOD_train_instances_to_flip = int(len(ICMP_FLOOD_train_instances) * n)
    # Flip randomly n% of ICMP_FLOOD to Benign in training set
    ICMP_FLOOD_to_Benign_flipping = random.sample(ICMP_FLOOD_train_instances,
        ICMP_FLOOD_train_instances_to_flip)
    poisoned_dataset.loc[ICMP_FLOOD_to_Benign_flipping, "Mapped_Label"] = "Benign"
    # Map the categorical labels into numerical for training
    train_labels = poisoned_dataset.loc[train_indices, "Mapped_Label"].astype('category').cat.set_categories(class_names).cat.codes
    # Map the categorical labels into numerical for evaluation
    test_labels = dataset.loc[test_indices,"Mapped_Label"].astype('category').cat.set_categories(class_names).cat.codes

    # Train model
    model = DecisionTreeClassifier(random_state=42)
    # The parameter random_state=42 simply fixes the seed for
    # reproducible results
    model.fit(X_train,train_labels)

    # Predict on the test dataset
    predicted_labels = model.predict(X_test)
    predictions.append(predicted_labels)

    # Evaluate metrics
    accuracy = accuracy_score(test_labels, predicted_labels)
    precision = precision_score(test_labels, predicted_labels, average='macro', zero_division=0)
    recall = recall_score(test_labels, predicted_labels, average='macro', zero_division=0)
    f1 = f1_score(test_labels, predicted_labels, average='macro', zero_division=0)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    if (n) == 0:
      print(f"Clean Dataset → Acc: {accuracy:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}, F1: {f1:.4f}")
    else:
      print(f"Poisoning {int(n*100)}% → Acc: {accuracy:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}, F1: {f1:.4f}")

    #Remove (''') in order to print the confusion matrix
    '''
    # Confusion Matrix
    cm = confusion_matrix(test_labels, predicted_labels)

    # Optionally: Pretty print using seaborn heatmap
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title(f"Confusion Matrix (Poisoning {int(n*100)}%)")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()
    plt.close()
    '''


Clean Dataset → Acc: 0.9979, Prec: 0.9543, Rec: 0.9557, F1: 0.9549
Poisoning 10% → Acc: 0.9875, Prec: 0.9312, Rec: 0.9454, F1: 0.9362
Poisoning 20% → Acc: 0.9749, Prec: 0.9180, Rec: 0.9421, F1: 0.9236
Poisoning 30% → Acc: 0.9573, Prec: 0.9062, Rec: 0.9386, F1: 0.9099
Poisoning 40% → Acc: 0.9350, Prec: 0.8924, Rec: 0.9264, F1: 0.8895
Poisoning 50% → Acc: 0.9104, Prec: 0.8940, Rec: 0.9165, F1: 0.8775


🤔Q2: Observe the results of the previous question and explain why the accuracy is still high even though the poisoning rate is 50%?

🔎Q3: Run the code below to calculate the Attack Success Rate (ASR) for the ICMP flood class for each poisoning rate (from 10% to 50%).📈

*Listing 2*

In [ ]:
ICMP_FLOOD_test_instances = test_indices[dataset.loc[test_indices, "Mapped_Label"] == "ICMP_FLOOD"]
ICMP_FLOOD_test_instances_index = test_indices.get_indexer_for(ICMP_FLOOD_test_instances)
# We use the predicted_lables from Listing 1
for n, item in zip(poisoning_rate, predictions):
  misclassified_ICMP_FLOOD_instances = np.sum(item[ICMP_FLOOD_test_instances_index] == class_names.index("Benign"))
  total_ICMP_FLOOD_test_instances=len(ICMP_FLOOD_test_instances_index) if len(ICMP_FLOOD_test_instances_index) > 0 else 0
  #  Calculate ASR
  asr = misclassified_ICMP_FLOOD_instances / total_ICMP_FLOOD_test_instances
  print(f"Poisoning {int(n*100)}% → ASR: {asr:.4f}")

Poisoning 0% → ASR: 0.0000
Poisoning 10% → ASR: 0.0679
Poisoning 20% → ASR: 0.1468
Poisoning 30% → ASR: 0.2537
Poisoning 40% → ASR: 0.4064
Poisoning 50% → ASR: 0.5530


***DEFENSIVE EXERCISES***

🤔Q1: Changing which hyperparameters can make the decision tree model more robust to poisoning attacks? (💡Hint: Two are needed)

🔎Q2: Retrain in order to evaluate the robustness of the model.🛡️

*Listing 3*

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


input_file_path="/content/Exercise_Dataset/exercise.csv"
dataset = pd.read_csv(input_file_path, sep=",", decimal=".")
# Prepare data for training and testing
X = dataset[['Tot size', 'syn_flag_number','Variance', 'TCP', 'ICMP', 'ack_flag_number','Header_Length', 'psh_flag_number', 'IAT', 'UDP','fin_flag_number', 'rst_flag_number', 'Std']]
y = dataset['Mapped_Label'].astype('category')
class_names = y.cat.categories.tolist()

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_indices = X_train.index
test_indices = X_test.index

# Initialize list for results

predictions = []
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []


poisoning_rate = np.arange(0, 0.6, 0.1)
# Run through different poisoning rate percentages
for n in poisoning_rate:
    poisoned_dataset = dataset.copy()
    # Select all the ICMP_FLOOD instances in training set
    ICMP_FLOOD_train_instances = poisoned_dataset[(poisoned_dataset["Mapped_Label"] == "ICMP_FLOOD") &
    (poisoned_dataset.index.isin(train_indices))].index.tolist()

    # Find the proportion of the IMCP_FLOOD class to be flipped
    ICMP_FLOOD_train_instances_to_flip = int(len(ICMP_FLOOD_train_instances) * n)
    # Flip randomly n% of ICMP_FLOOD to Benign in training set
    ICMP_FLOOD_to_Benign_flipping = random.sample(ICMP_FLOOD_train_instances,
        ICMP_FLOOD_train_instances_to_flip)
    poisoned_dataset.loc[ICMP_FLOOD_to_Benign_flipping, "Mapped_Label"] = "Benign"
    # Map the categorical labels into numerical for training
    train_labels = poisoned_dataset.loc[train_indices, "Mapped_Label"].astype('category').cat.set_categories(class_names).cat.codes
    # Map the categorical labels into numerical for evaluation
    test_labels = dataset.loc[test_indices,"Mapped_Label"].astype('category').cat.set_categories(class_names).cat.codes

    # Train model
    model = DecisionTreeClassifier(random_state=42, max_depth=20,  class_weight="balanced")
    # The parameter random_state=42 simply fixes the seed for
    # reproducible results
    model.fit(X_train,train_labels)

    # Predict on the test dataset
    predicted_labels = model.predict(X_test)
    predictions.append(predicted_labels)

    # Evaluate metrics
    accuracy = accuracy_score(test_labels, predicted_labels)
    precision = precision_score(test_labels, predicted_labels, average='macro', zero_division=0)
    recall = recall_score(test_labels, predicted_labels, average='macro', zero_division=0)
    f1 = f1_score(test_labels, predicted_labels, average='macro', zero_division=0)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    if (n) == 0:
      print(f"Clean Dataset → Acc: {accuracy:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}, F1: {f1:.4f}")
    else:
      print(f"Poisoning {int(n*100)}% → Acc: {accuracy:.4f}, Prec: {precision:.4f}, Rec: {recall:.4f}, F1: {f1:.4f}")

    #Remove (''') in order to print the confusion matrix
    '''
    # Confusion Matrix
    cm = confusion_matrix(test_labels, predicted_labels)

    # Optionally: Pretty print using seaborn heatmap
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title(f"Confusion Matrix Robust(Poisoning {int(n*100)}%)")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.show()
    plt.close()
    '''

Clean Dataset → Acc: 0.9978, Prec: 0.9437, Rec: 0.9600, F1: 0.9499
Poisoning 10% → Acc: 0.9969, Prec: 0.9397, Rec: 0.9536, F1: 0.9449
Poisoning 20% → Acc: 0.9955, Prec: 0.9425, Rec: 0.9508, F1: 0.9456
Poisoning 30% → Acc: 0.9963, Prec: 0.9467, Rec: 0.9514, F1: 0.9482
Poisoning 40% → Acc: 0.9942, Prec: 0.9320, Rec: 0.9578, F1: 0.9418
Poisoning 50% → Acc: 0.9964, Prec: 0.9367, Rec: 0.9558, F1: 0.9440


🔎Q3: Recalculate the ASR values to validate the robustness of the model.🛡️

*Listing 4*

In [ ]:
ICMP_FLOOD_test_instances = test_indices[dataset.loc[test_indices, "Mapped_Label"] == "ICMP_FLOOD"]
ICMP_FLOOD_test_instances_index = test_indices.get_indexer_for(ICMP_FLOOD_test_instances)
# We use the predicted_lables from Listing 1
for n, item in zip(poisoning_rate, predictions):
  misclassified_ICMP_FLOOD_instances = np.sum(item[ICMP_FLOOD_test_instances_index] == class_names.index("Benign"))
  total_ICMP_FLOOD_test_instances=len(ICMP_FLOOD_test_instances_index) if len(ICMP_FLOOD_test_instances_index) > 0 else 0
  #  Calculate ASR
  asr = misclassified_ICMP_FLOOD_instances / total_ICMP_FLOOD_test_instances
  print(f"Poisoning {int(n*100)}% → ASR: {asr:.4f}")

Poisoning 0% → ASR: 0.0000
Poisoning 10% → ASR: 0.0063
Poisoning 20% → ASR: 0.0114
Poisoning 30% → ASR: 0.0048
Poisoning 40% → ASR: 0.0121
Poisoning 50% → ASR: 0.0213
